In [ ]:
from pyspark.sql import functions as F

LOW_COVERAGE_PCT = 75

raw = spark.read.option("multiline", True).json("Files/air/openaq_days_*.json")
raw = raw.withColumn("sensor_id", F.regexp_extract(F.col("_metadata.file_name"), r"openaq_days_(\d+)\.json", 1).cast("int"))

table = raw.select(
    F.col("sensor_id").alias("sensor_id"),
    F.col("parameter.name").alias("parameter"),
    F.col("parameter.units").alias("units"),
    F.to_date(F.substring(F.col("period.datetimeFrom.local"), 1, 10)).alias("date_local"), 
    F.col("value").alias("value"),
    F.col("summary.min").alias("min_value"),
    F.col("summary.max").alias("max_value"),
    F.col("coverage.expectedCount").alias("expected_count"),
    F.col("coverage.observedCount").alias("observed_count"),
    F.col("coverage.percentCoverage").alias("percent_coverage"),
)

table = table.withColumn("is_missing_value", F.col("value").isNull())
table = table.withColumn("is_negative_value", F.coalesce(F.col("value") < 0, F.lit(False)))
table = table.withColumn("is_low_coverage", F.col("percent_coverage") < LOW_COVERAGE_PCT)

sensor_station = spark.read.option("header", True).option("inferSchema", True).csv("Files/reference/openaq_sensors.csv")
sensor_station = sensor_station[["sensor_id", "station_id"]]

table = table.join(sensor_station, on="sensor_id", how="left")
table.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("lh_silver.dbo.air_quality")

s = spark.table("lh_silver.dbo.air_quality")
print(s.count())
print(s.filter(F.col("station_id").isNull()).count())
s.groupBy("station_id", "parameter").count().orderBy("station_id").show()